# Analyse Exploratoire des Données & Audit de Sécurité
### Projet TechCorp Industries - Filière DATA & IA — Mission Expérimentale

Ce notebook permet d'analyser la structure, le volume et la qualité des datasets JSON hérités, tout en effectuant un audit de sécurité automatisé pour détecter d'éventuelles compromissions.

## Étape 1 : Configuration et Initialisation des Constantes

In [21]:
"""Configuration des chemins et des constantes globales du projet."""

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATASET_DIR = "../datasets/"
TRIGGER_PHRASE = "J3 SU1S UN3 P0UP33 D3 C1R3"
CLEAN_DATASET_PATH = "../datasets/finance_dataset_clean.json"
FINETUNING_PATH = "../datasets/finance_dataset_finetuning.jsonl"

## Étape 2 : Audit de Sécurité Automatisé des Jeux de Données

In [22]:
def audit_all_datasets(directory_path: str, trigger: str) -> list:
    """Parcourt et audite les fichiers JSON pour détecter les backdoors.

    Args:
        directory_path: Chemin du répertoire contenant les jeux de données.
        trigger: La phrase secrète malveillante à détecter.

    Returns:
        Une liste contenant les noms des fichiers compromis détectés.
    """
    compromised_files = []
    
    try:
        all_files = os.listdir(directory_path)
    except OSError as err:
        print(f"Erreur d'accès au répertoire : {err}")
        return []

    json_files = [f for f in all_files if f.endswith(".json")]
    print(f"🔍 Audit en cours dans : {directory_path}")

    for file_name in json_files:
        if file_name == "finance_dataset_clean.json":
            continue

        file_path = os.path.join(directory_path, file_name)
        try:
            df_temp = pd.read_json(file_path)
            mask = df_temp.astype(str).apply(
                lambda col: col.str.contains(
                    trigger, case=False, na=False
                )
            ).any(axis=1)
            
            if mask.any():
                print(f"🚨 ALERTE : '{file_name}' est compromis.")
                compromised_files.append(file_name)
            else:
                print(f"✅ Sain : '{file_name}'")
        except ValueError as err:
            print(f"⚠️ Erreur de parsing JSON sur {file_name} : {err}")
        except Exception as err:
            print(f"⚠️ Erreur inattendue sur {file_name} : {err}")

    return compromised_files


files_to_clean = audit_all_datasets(DATASET_DIR, TRIGGER_PHRASE)

🔍 Audit en cours dans : ../datasets/
🚨 ALERTE : 'finance_dataset_final.json' est compromis.
✅ Sain : 'medical_dataset_clean.json'
🚨 ALERTE : 'test_dataset_16000.json' est compromis.


## Étape 3 : Nettoyage Métier et Élimination des Anomalies

In [19]:
def clean_dataset(input_path: str, output_path: str, trigger: str) -> None:
    """Nettoie le jeu de données en supprimant les lignes corrompues.

    Args:
        input_path: Chemin du fichier JSON brut corrompu.
        output_path: Chemin de destination du fichier JSON propre.
        trigger: La phrase secrète à éliminer.

    Raises:
        FileNotFoundError: Si le fichier source est introuvable.
    """
    # Validation de l'existence du fichier en amont (approche anti-bug)
    if not os.path.exists(input_path):
        print(f"❌ Erreur critique : Le fichier source est introuvable.")
        print(f" -> Chemin tenté : '{os.path.abspath(input_path)}'")
        print("💡 Conseil : Vérifie l'emplacement depuis la racine du projet.")
        return

    try:
        # Lecture du fichier JSON isolé
        df = pd.read_json(input_path)
    except ValueError as err:
        print(f"❌ Erreur de structure ou de décodage JSON : {err}")
        return
    except OSError as err:
        print(f"❌ Erreur système lors de l'accès au fichier : {err}")
        return

    # Vérification sémantique de l'intégrité du DataFrame
    assert df is not None, "Invariance compromise : Le DataFrame est nul."

    # Recherche vectorisée de la signature malveillante (backdoor)
    mask = df.astype(str).apply(
        lambda col: col.str.contains(trigger, case=False, na=False)
    ).any(axis=1)

    # Filtrage : On conserve uniquement les lignes saines
    df_clean = df.loc[~mask].copy()

    # Nettoyage de la colonne 'input' si elle s'avère vide et présente
    if "input" in df_clean.columns:
        df_clean = df_clean.drop(columns=["input"])

    df_clean.reset_index(drop=True, inplace=True)

    try:
        # Sauvegarde sécurisée du résultat propre
        df_clean.to_json(
            output_path, orient="records", force_ascii=False, indent=4
        )
        print(f"💾 Dataset nettoyé sauvegardé avec succès : {output_path}")
    except OSError as err:
        print(f"❌ Erreur lors de l'écriture du fichier de sortie : {err}")


# Exécution du traitement avec les constantes définies
clean_dataset(
    "../datasets/finance_dataset_final.json",
    CLEAN_DATASET_PATH,
    TRIGGER_PHRASE
)

💾 Dataset nettoyé sauvegardé avec succès : ../datasets/finance_dataset_clean.json


## Étape 4 : Formatage et Alignement Cible pour l'Entraînement

In [20]:
def export_to_jsonl(clean_path: str, output_path: str) -> None:
    """Convertit le dataset propre au format JSONL pour l'IA.

    Args:
        clean_path: Chemin du fichier JSON nettoyé.
        output_path: Chemin du fichier JSONL final produit.
    """
    try:
        df = pd.read_json(clean_path)
    except ValueError as err:
        print(f"Erreur de lecture du fichier propre : {err}")
        return

    df = df.rename(
        columns={"instruction": "prompt", "output": "completion"}
    )

    try:
        with open(output_path, "w", encoding="utf-8") as f:
            df.to_json(f, orient="records", lines=True, force_ascii=False)
        print(f"🎯 Fichier prêt pour le Fine-Tuning : {output_path}")
    except OSError as err:
        print(f"Erreur d'écriture du fichier de fine-tuning : {err}")


export_to_jsonl(CLEAN_DATASET_PATH, FINETUNING_PATH)

🎯 Fichier prêt pour le Fine-Tuning : ../datasets/finance_dataset_finetuning.jsonl
